# 01 - Análisis exploratorio y preprocesado

## Práctica B4-T2 - XAI para concesión de crédito

Este notebook prepara la base de trabajo para el resto del proyecto. La idea es dejar el dataset limpio, documentado y con decisiones de preprocesado justificadas antes de pasar al modelado.

Trabajamos con dos ficheros:

- `cs_construccion.csv`: datos con variable objetivo conocida. Se usa para análisis, validación y entrenamiento.
- `cs_produccion.csv`: datos sin etiqueta real. Se usa al final para generar las predicciones entregables.

La variable objetivo es `SeriousDlqin2yrs`, que indica si una persona tuvo una morosidad grave de 90 días o más en los dos últimos años.

### Qué se hace en este notebook

1. Carga y revisión de datos.
2. Lectura del diccionario de variables.
3. Análisis de la variable objetivo y del desbalanceo.
4. Análisis de valores perdidos.
5. Imputación mediante KNN reutilizable para construcción y producción.
6. Análisis exhaustivo de valores atípicos.
7. Tratamiento de outliers mediante clipping/winsorización.
8. Análisis de distribuciones.
9. Análisis de correlaciones Pearson y Spearman.
10. Análisis PCA.
11. Generación de datasets preprocesados para los notebooks posteriores.

> Importante: todos los objetos que aprenden parámetros, como imputadores, límites de clipping o escaladores, se ajustan únicamente con `cs_construccion.csv`. Después se aplican a producción. Esto evita leakage.

In [ ]:
# ==============================
# Imports generales
# ==============================

from __future__ import annotations

import json
import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Carpeta de salida común para todos los notebooks.
OUTPUT_DIR = Path("outputs")
PLOTS_DIR = OUTPUT_DIR / "plots"
OBJECTS_DIR = OUTPUT_DIR / "objects"

for path in [OUTPUT_DIR, PLOTS_DIR, OBJECTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## 1. Localización y carga de ficheros

La siguiente función busca los CSV en varias ubicaciones posibles. Así el notebook puede ejecutarse tanto desde la carpeta raíz del proyecto como desde una carpeta de notebooks.

Se prioriza la carpeta `data/`, que es donde se incluyen los datos dentro del ZIP.

In [ ]:
def find_file(filename: str) -> Path:
    """
    Busca un fichero en rutas habituales del proyecto.

    Esto hace el notebook más robusto: no depende de que se ejecute exactamente
    desde una carpeta concreta.
    """
    candidates = [
        Path.cwd() / "data" / filename,
        Path.cwd() / filename,
        Path.cwd().parent / "data" / filename,
        Path.cwd().parent / filename,
        Path("/mnt/data") / filename,  # útil en este entorno, no molesta fuera
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"No se ha encontrado {filename}. Rutas probadas:\n" +
        "\n".join(str(c) for c in candidates)
    )


train_path = find_file("cs_construccion.csv")
prod_path = find_file("cs_produccion.csv")
dictionary_path = find_file("DataDictionary.csv")

print("Fichero construcción:", train_path)
print("Fichero producción:", prod_path)
print("Diccionario:", dictionary_path)

raw_train = pd.read_csv(train_path)
raw_prod = pd.read_csv(prod_path)

# El diccionario viene separado por punto y coma.
data_dictionary = pd.read_csv(dictionary_path, sep=";")

print("Construcción:", raw_train.shape)
print("Producción:", raw_prod.shape)

In [ ]:
# Primer vistazo al dataset de construcción
raw_train.head()

In [ ]:
# Primer vistazo al dataset de producción
raw_prod.head()

In [ ]:
# Diccionario de variables
# Esta tabla se usará como apoyo para interpretar cada variable.
data_dictionary

## 2. Revisión básica de estructura

Antes de transformar datos conviene responder preguntas simples:

- ¿Cuántas filas y columnas hay?
- ¿Qué tipos de datos tenemos?
- ¿La variable objetivo está en construcción y vacía en producción?
- ¿Existen columnas constantes?
- ¿Hay duplicados exactos?

Aunque parezcan comprobaciones básicas, suelen detectar muchos errores antes de construir modelos.

In [ ]:
TARGET = "SeriousDlqin2yrs"
FEATURES = [c for c in raw_train.columns if c != TARGET]

print("Variable objetivo:", TARGET)
print("Número de variables explicativas:", len(FEATURES))
print(FEATURES)

basic_info = pd.DataFrame({
    "dtype_train": raw_train.dtypes.astype(str),
    "dtype_prod": raw_prod.dtypes.astype(str),
    "missing_train": raw_train.isna().sum(),
    "missing_prod": raw_prod.isna().sum(),
    "n_unique_train": raw_train.nunique(dropna=True),
    "n_unique_prod": raw_prod.nunique(dropna=True),
})

basic_info["missing_train_pct"] = 100 * basic_info["missing_train"] / len(raw_train)
basic_info["missing_prod_pct"] = 100 * basic_info["missing_prod"] / len(raw_prod)

basic_info.sort_values("missing_train_pct", ascending=False)

In [ ]:
print("Duplicados exactos en construcción:", raw_train.duplicated().sum())
print("Duplicados exactos en producción:", raw_prod.duplicated().sum())

constant_cols_train = [c for c in raw_train.columns if raw_train[c].nunique(dropna=False) <= 1]
constant_cols_prod = [c for c in raw_prod.columns if raw_prod[c].nunique(dropna=False) <= 1]

print("Columnas constantes en construcción:", constant_cols_train)
print("Columnas constantes en producción:", constant_cols_prod)

## 3. Variable objetivo: `SeriousDlqin2yrs`

Esta variable vale:

- `0`: el cliente no tuvo morosidad grave de 90 días o más en los dos últimos años.
- `1`: el cliente sí tuvo morosidad grave.

En problemas de riesgo de crédito suele haber mucho desbalanceo: la mayoría de clientes no incumple. Esto afecta al modelado porque una red neuronal podría aprender a predecir casi siempre la clase mayoritaria. Por eso más adelante no miraremos solo `accuracy`, sino también coste, recall, precision, F1, AUC y matriz de confusión.

In [ ]:
target_counts = raw_train[TARGET].value_counts(dropna=False).rename_axis(TARGET).reset_index(name="n")
target_counts["pct"] = 100 * target_counts["n"] / len(raw_train)
target_counts

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(target_counts[TARGET].astype(str), target_counts["n"])
plt.title("Distribución de la variable objetivo")
plt.xlabel("SeriousDlqin2yrs")
plt.ylabel("Número de observaciones")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 4. Estadísticos descriptivos iniciales

La tabla siguiente resume posición, dispersión y percentiles. En este dataset es especialmente importante mirar `p99` y `max`, porque varias variables financieras suelen tener valores extremos que deforman los histogramas.

In [ ]:
def describe_numeric(df: pd.DataFrame, columns: Optional[List[str]] = None) -> pd.DataFrame:
    """
    Devuelve un resumen ampliado para variables numéricas.

    Incluye percentiles altos para detectar concentración a la izquierda y colas largas.
    """
    if columns is None:
        columns = df.select_dtypes(include="number").columns.tolist()

    rows = []
    for col in columns:
        s = df[col].dropna()
        rows.append({
            "variable": col,
            "n": len(s),
            "missing": df[col].isna().sum(),
            "missing_pct": 100 * df[col].isna().mean(),
            "min": s.min() if len(s) else np.nan,
            "p01": s.quantile(0.01) if len(s) else np.nan,
            "p05": s.quantile(0.05) if len(s) else np.nan,
            "p25": s.quantile(0.25) if len(s) else np.nan,
            "median": s.median() if len(s) else np.nan,
            "mean": s.mean() if len(s) else np.nan,
            "p75": s.quantile(0.75) if len(s) else np.nan,
            "p95": s.quantile(0.95) if len(s) else np.nan,
            "p99": s.quantile(0.99) if len(s) else np.nan,
            "max": s.max() if len(s) else np.nan,
            "std": s.std() if len(s) else np.nan,
            "skew": s.skew() if len(s) else np.nan,
            "kurtosis": s.kurtosis() if len(s) else np.nan,
        })

    return pd.DataFrame(rows).sort_values("variable").reset_index(drop=True)

numeric_summary_raw = describe_numeric(raw_train)
numeric_summary_raw

In [ ]:
numeric_summary_raw.to_csv(OUTPUT_DIR / "numeric_summary_raw.csv", index=False)
print("Guardado:", OUTPUT_DIR / "numeric_summary_raw.csv")

## 5. Análisis de valores perdidos

En este dataset los valores perdidos aparecen principalmente en variables de renta y dependientes. No deben eliminarse sin más: perderíamos muchas filas y además el hecho de que falte renta puede ser informativo.

Por eso haremos dos cosas:

1. Crear indicadores binarios de ausencia (`*_was_missing`).
2. Imputar los valores usando KNN, ajustado únicamente en construcción y aplicado después a producción.

In [ ]:
def missing_report(train: pd.DataFrame, prod: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """Crea un informe comparando missing en construcción y producción."""
    report = pd.DataFrame({
        "missing_train": train.isna().sum(),
        "missing_train_pct": 100 * train.isna().mean(),
    })

    if prod is not None:
        report["missing_prod"] = prod.isna().sum()
        report["missing_prod_pct"] = 100 * prod.isna().mean()

    return report.sort_values("missing_train_pct", ascending=False)

missing = missing_report(raw_train, raw_prod)
missing

In [ ]:
# Visualización sencilla del porcentaje de missing por variable.
missing_nonzero = missing[missing["missing_train_pct"] > 0].copy()

plt.figure(figsize=(10, 4))
plt.bar(missing_nonzero.index, missing_nonzero["missing_train_pct"])
plt.title("Porcentaje de valores perdidos en construcción")
plt.xlabel("Variable")
plt.ylabel("Missing (%)")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Analizamos si tener missing está asociado al target.
# Esto ayuda a decidir si debemos añadir flags de missing.
missing_effect_rows = []

for col in FEATURES:
    if raw_train[col].isna().any():
        flag = raw_train[col].isna()
        missing_effect_rows.append({
            "variable": col,
            "n_missing": flag.sum(),
            "default_rate_missing": raw_train.loc[flag, TARGET].mean(),
            "default_rate_not_missing": raw_train.loc[~flag, TARGET].mean(),
            "difference": raw_train.loc[flag, TARGET].mean() - raw_train.loc[~flag, TARGET].mean(),
        })

missing_effect = pd.DataFrame(missing_effect_rows)
missing_effect

## 6. Imputación KNN reutilizable

### Por qué KNN

La imputación KNN rellena un valor ausente buscando clientes parecidos según el resto de variables. Por ejemplo, para imputar `MonthlyIncome`, no usa una media global, sino valores de clientes con perfiles similares.

### Problema práctico

KNN depende de distancias. Si una variable está en miles de euros y otra en unidades, la variable con escala mayor domina la distancia. Por eso implementamos un imputador que:

1. Ajusta un `StandardScaler` con los datos de construcción.
2. Aplica KNN sobre variables escaladas.
3. Deshace el escalado para volver a las unidades originales.

Además, se guarda el objeto para poder aplicarlo después a producción y a cualquier scoring futuro.

In [ ]:
@dataclass
class ScaledKNNImputer:
    """
    Imputador KNN con escalado interno y muestreo para datasets grandes.

    KNNImputer puede ser costoso si se ajusta contra todas las filas. Para que el
    notebook sea ejecutable en portátil, el imputador se ajusta con una muestra
    representativa de construcción y solo transforma las filas que realmente
    tienen algún valor perdido. Las filas completas se dejan intactas.

    - fit: aprende medias/desviaciones y vecinos usando construcción.
    - transform: imputa únicamente filas con missing.
    - salida: conserva las unidades originales, porque se deshace el escalado.
    """
    columns: List[str]
    n_neighbors: int = 7
    weights: str = "distance"
    fit_sample_size: int = 10000
    random_state: int = 42

    def __post_init__(self):
        self.scaler = StandardScaler()
        self.imputer = KNNImputer(n_neighbors=self.n_neighbors, weights=self.weights)
        self.fitted_ = False
        self.fit_indices_ = None

    def fit(self, df: pd.DataFrame):
        x_all = df[self.columns].astype(float).copy()

        # StandardScaler ignora NaN al calcular media/std en versiones recientes de sklearn.
        self.scaler.fit(x_all)

        # Usamos una muestra para que KNN sea viable en datasets grandes.
        # Siempre intentamos incluir filas con missing, porque son las más relevantes para aprender imputación.
        rng = np.random.default_rng(self.random_state)
        missing_mask = x_all.isna().any(axis=1)
        missing_indices = x_all.index[missing_mask].to_numpy()
        complete_indices = x_all.index[~missing_mask].to_numpy()

        n_missing_keep = min(len(missing_indices), self.fit_sample_size // 2)
        n_complete_keep = max(self.fit_sample_size - n_missing_keep, 0)

        sampled_missing = rng.choice(missing_indices, size=n_missing_keep, replace=False) if n_missing_keep > 0 else np.array([], dtype=int)
        sampled_complete = rng.choice(complete_indices, size=min(n_complete_keep, len(complete_indices)), replace=False) if len(complete_indices) > 0 and n_complete_keep > 0 else np.array([], dtype=int)

        fit_indices = np.concatenate([sampled_missing, sampled_complete])
        if len(fit_indices) == 0:
            fit_indices = x_all.index.to_numpy()

        self.fit_indices_ = fit_indices
        x_fit = x_all.loc[fit_indices]
        x_fit_scaled = self.scaler.transform(x_fit)
        self.imputer.fit(x_fit_scaled)
        self.fitted_ = True
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        if not self.fitted_:
            raise RuntimeError("El imputador debe ajustarse con fit antes de usar transform.")

        result = df.copy()
        # Convertimos las columnas imputables a float para evitar conflictos de dtype
        # al insertar valores imputados que pueden venir con decimales.
        result[self.columns] = result[self.columns].astype(float)
        x = result[self.columns].astype(float).copy()
        rows_with_missing = x.isna().any(axis=1)

        if rows_with_missing.any():
            x_missing = x.loc[rows_with_missing]
            x_scaled = self.scaler.transform(x_missing)
            x_imputed_scaled = self.imputer.transform(x_scaled)
            x_imputed = self.scaler.inverse_transform(x_imputed_scaled)
            result.loc[rows_with_missing, self.columns] = x_imputed

        return result

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).transform(df)


def add_missing_flags(train: pd.DataFrame, prod: pd.DataFrame, columns: Optional[List[str]] = None) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Añade columnas binarias indicando si el valor original era missing.

    Estas columnas se crean antes de imputar, porque después de imputar ya no sabríamos
    dónde estaban los huecos originales.
    """
    if columns is None:
        columns = [c for c in train.columns if train[c].isna().any() or prod[c].isna().any()]

    train_out = train.copy()
    prod_out = prod.copy()
    flag_cols = []

    for col in columns:
        flag_col = f"{col}_was_missing"
        train_out[flag_col] = train_out[col].isna().astype(int)
        prod_out[flag_col] = prod_out[col].isna().astype(int)
        flag_cols.append(flag_col)

    return train_out, prod_out, flag_cols

In [ ]:
# Creamos copias de trabajo.
train = raw_train.copy()
prod = raw_prod.copy()

# En producción la variable objetivo aparece vacía; no se utiliza como feature.
prod[TARGET] = np.nan

# Añadimos flags de missing usando construcción y producción.
train, prod, missing_flag_cols = add_missing_flags(train, prod, columns=[c for c in FEATURES if raw_train[c].isna().any() or raw_prod[c].isna().any()])

print("Flags de missing creados:", missing_flag_cols)

# Columnas que se imputan con KNN: variables explicativas originales.
impute_cols = FEATURES.copy()

knn_imputer = ScaledKNNImputer(columns=impute_cols, n_neighbors=7, weights="distance", fit_sample_size=10000, random_state=RANDOM_STATE)
train_imputed = knn_imputer.fit_transform(train)
prod_imputed = knn_imputer.transform(prod)

print("Missing tras imputación en construcción:", train_imputed[impute_cols].isna().sum().sum())
print("Missing tras imputación en producción:", prod_imputed[impute_cols].isna().sum().sum())

In [ ]:
# Algunas variables deberían ser conteos enteros.
# Después de KNN pueden quedar decimales. Para mantener sentido financiero,
# redondeamos las variables de conteo y evitamos valores negativos.
count_cols = [
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
]

for col in count_cols:
    if col in train_imputed.columns:
        train_imputed[col] = np.round(train_imputed[col]).clip(lower=0)
        prod_imputed[col] = np.round(prod_imputed[col]).clip(lower=0)

# La edad no puede ser 0 en un dataset real de crédito. No eliminamos todavía:
# lo analizamos como posible outlier o error de captura.
train_imputed[count_cols].head()

## 7. Análisis exhaustivo de valores atípicos

En variables financieras es normal encontrar colas largas. El problema es que valores extremos pueden:

- distorsionar histogramas;
- afectar a escaladores basados en media y desviación típica;
- dificultar el entrenamiento de una red neuronal;
- dominar distancias en KNN o PCA.

No todos los outliers son errores: algunos representan clientes extremos reales. Por eso primero los cuantificamos y después aplicamos un tratamiento conservador.

In [ ]:
def outlier_report(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    """
    Calcula indicadores de outliers por IQR y percentiles extremos.
    """
    rows = []
    for col in columns:
        s = df[col].dropna().astype(float)
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower_iqr = q1 - 1.5 * iqr
        upper_iqr = q3 + 1.5 * iqr
        p001 = s.quantile(0.001)
        p01 = s.quantile(0.01)
        p99 = s.quantile(0.99)
        p999 = s.quantile(0.999)

        rows.append({
            "variable": col,
            "min": s.min(),
            "p001": p001,
            "p01": p01,
            "q1": q1,
            "median": s.median(),
            "q3": q3,
            "p99": p99,
            "p999": p999,
            "max": s.max(),
            "iqr": iqr,
            "lower_iqr": lower_iqr,
            "upper_iqr": upper_iqr,
            "n_outliers_iqr": int(((s < lower_iqr) | (s > upper_iqr)).sum()),
            "pct_outliers_iqr": 100 * ((s < lower_iqr) | (s > upper_iqr)).mean(),
            "n_above_p99": int((s > p99).sum()),
            "pct_above_p99": 100 * (s > p99).mean(),
        })

    return pd.DataFrame(rows).sort_values("pct_outliers_iqr", ascending=False)

outliers_raw = outlier_report(train_imputed, FEATURES)
outliers_raw

In [ ]:
outliers_raw.to_csv(OUTPUT_DIR / "outlier_report_before_clipping.csv", index=False)
print("Guardado:", OUTPUT_DIR / "outlier_report_before_clipping.csv")

### 7.1 Visualización de distribuciones reales

En los histogramas originales muchas variables aparecen pegadas a la izquierda porque existen valores máximos enormes. Para verlo bien, por cada variable mostramos:

1. Histograma completo.
2. Histograma recortado al percentil 99.
3. Histograma con escala logarítmica en el eje Y.
4. Boxplot horizontal.

Esto permite distinguir entre la distribución de la mayoría de clientes y los casos extremos.

In [ ]:
def plot_distribution_diagnostics(df: pd.DataFrame, columns: List[str], bins: int = 50, upper_quantile: float = 0.99):
    """
    Dibuja diagnóstico de distribución para cada variable.

    Se usa un gráfico por variable para evitar paneles enormes difíciles de leer.
    """
    for col in columns:
        s = df[col].dropna().astype(float)
        upper = s.quantile(upper_quantile)
        s_zoom = s[s <= upper]

        fig, axes = plt.subplots(1, 4, figsize=(22, 4))

        axes[0].hist(s, bins=bins, edgecolor="black")
        axes[0].set_title(f"{col}\nCompleto")
        axes[0].set_xlabel(col)
        axes[0].set_ylabel("Frecuencia")
        axes[0].grid(alpha=0.25)

        axes[1].hist(s_zoom, bins=bins, edgecolor="black")
        axes[1].set_title(f"{col}\nZoom <= p{int(upper_quantile * 100)}")
        axes[1].set_xlabel(col)
        axes[1].set_ylabel("Frecuencia")
        axes[1].grid(alpha=0.25)

        axes[2].hist(s, bins=bins, edgecolor="black")
        axes[2].set_yscale("log")
        axes[2].set_title(f"{col}\nY log")
        axes[2].set_xlabel(col)
        axes[2].set_ylabel("Frecuencia log")
        axes[2].grid(alpha=0.25)

        axes[3].boxplot(s, vert=False)
        axes[3].set_title(f"{col}\nBoxplot")
        axes[3].set_xlabel(col)
        axes[3].grid(alpha=0.25)

        plt.tight_layout()
        plt.show()

# Si quieres ejecutar solo algunas variables, cambia FEATURES por una lista, por ejemplo:
# ["DebtRatio", "MonthlyIncome", "RevolvingUtilizationOfUnsecuredLines"]
plot_distribution_diagnostics(train_imputed, FEATURES)

### 7.2 Decisión de tratamiento de outliers

Para una red neuronal y un bandit contextual conviene evitar que valores extremos dominen el aprendizaje. Usaremos un tratamiento conservador:

- No eliminamos filas.
- Ajustamos límites de clipping con construcción.
- Aplicamos los mismos límites a construcción y producción.
- Usamos percentiles 0.1 % y 99.9 % para no destruir demasiada información.

Esto es una winsorización suave: reduce el impacto de extremos muy raros, pero conserva el orden y la mayoría de la variabilidad.

In [ ]:
def fit_clipping_limits(
    df: pd.DataFrame,
    columns: List[str],
    lower_q: float = 0.001,
    upper_q: float = 0.999,
) -> Dict[str, Tuple[float, float]]:
    """Aprende límites de clipping a partir del dataset de construcción."""
    limits = {}
    for col in columns:
        s = df[col].dropna().astype(float)
        lower = float(s.quantile(lower_q))
        upper = float(s.quantile(upper_q))
        limits[col] = (lower, upper)
    return limits


def apply_clipping(df: pd.DataFrame, limits: Dict[str, Tuple[float, float]]) -> pd.DataFrame:
    """Aplica límites de clipping previamente ajustados."""
    out = df.copy()
    for col, (lower, upper) in limits.items():
        out[col] = out[col].clip(lower=lower, upper=upper)
    return out


# Variables sobre las que aplicamos clipping.
# Excluimos flags binarios de missing y el target.
clip_cols = FEATURES.copy()
clipping_limits = fit_clipping_limits(train_imputed, clip_cols, lower_q=0.001, upper_q=0.999)

train_clipped = apply_clipping(train_imputed, clipping_limits)
prod_clipped = apply_clipping(prod_imputed, clipping_limits)

pd.DataFrame([
    {"variable": col, "lower_limit": lim[0], "upper_limit": lim[1]}
    for col, lim in clipping_limits.items()
]).sort_values("variable")

In [ ]:
outliers_after = outlier_report(train_clipped, FEATURES)
outliers_after.to_csv(OUTPUT_DIR / "outlier_report_after_clipping.csv", index=False)
outliers_after

## 8. Feature engineering interpretativo

Creamos variables sencillas que suelen tener sentido financiero:

- `TotalPastDueEvents`: suma de retrasos de 30-59, 60-89 y 90+ días.
- `HasAnyPastDue`: indica si hubo cualquier retraso.
- `Has90DaysLate`: indica si hubo retrasos de 90+ días.
- `CreditLinesPerRealEstateLoan`: relación entre líneas abiertas totales e inmobiliarias.
- Transformaciones `log1p` para variables con colas largas.

Las variables originales se mantienen; las nuevas ayudan al modelo y también facilitan explicaciones posteriores.

In [ ]:
def add_credit_features(df: pd.DataFrame) -> pd.DataFrame:
    """Añade variables derivadas de riesgo crediticio."""
    out = df.copy()

    delinquency_cols = [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
    ]

    out["TotalPastDueEvents"] = out[delinquency_cols].sum(axis=1)
    out["HasAnyPastDue"] = (out["TotalPastDueEvents"] > 0).astype(int)
    out["Has90DaysLate"] = (out["NumberOfTimes90DaysLate"] > 0).astype(int)
    out["HasDependents"] = (out["NumberOfDependents"] > 0).astype(int)

    out["CreditLinesPerRealEstateLoan"] = (
        out["NumberOfOpenCreditLinesAndLoans"] / (1 + out["NumberRealEstateLoansOrLines"])
    )

    # log1p es log(1+x). Funciona con ceros y reduce la cola derecha.
    log_cols = [
        "RevolvingUtilizationOfUnsecuredLines",
        "DebtRatio",
        "MonthlyIncome",
        "NumberOfOpenCreditLinesAndLoans",
        "NumberRealEstateLoansOrLines",
        "TotalPastDueEvents",
    ]

    for col in log_cols:
        safe_values = out[col].clip(lower=0)
        out[f"{col}_log1p"] = np.log1p(safe_values)

    return out

train_fe = add_credit_features(train_clipped)
prod_fe = add_credit_features(prod_clipped)

# Features finales: todas menos target.
FINAL_FEATURES = [c for c in train_fe.columns if c != TARGET]

print("Número de features finales:", len(FINAL_FEATURES))
print(FINAL_FEATURES)

## 9. Relación de variables con la variable objetivo

Para variables numéricas, una forma útil de entender su relación con el riesgo es calcular la tasa de default por deciles. Así se ve si el riesgo aumenta o disminuye de forma ordenada.

Esta sección no es modelado todavía. Es análisis descriptivo para entender qué puede aprender el modelo.

In [ ]:
def default_rate_by_bins(df: pd.DataFrame, feature: str, target: str = TARGET, q: int = 10) -> pd.DataFrame:
    """
    Calcula tasa de default por cuantiles de una variable.

    Si una variable tiene muchos valores repetidos, qcut puede crear menos grupos.
    """
    temp = df[[feature, target]].copy()
    temp["bin"] = pd.qcut(temp[feature], q=q, duplicates="drop")
    result = temp.groupby("bin", observed=True).agg(
        n=(target, "size"),
        default_rate=(target, "mean"),
        feature_min=(feature, "min"),
        feature_max=(feature, "max"),
    ).reset_index()
    result["default_rate_pct"] = 100 * result["default_rate"]
    return result


for feature in ["age", "RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome", "TotalPastDueEvents"]:
    dr = default_rate_by_bins(train_fe, feature)
    display(dr)

    plt.figure(figsize=(9, 4))
    plt.plot(range(len(dr)), dr["default_rate_pct"], marker="o")
    plt.title(f"Tasa de default por deciles: {feature}")
    plt.xlabel("Decil de la variable")
    plt.ylabel("Default rate (%)")
    plt.grid(alpha=0.3)
    plt.show()

## 10. Correlaciones

Calculamos dos tipos:

- **Pearson**: relación lineal.
- **Spearman**: relación monótona, más robusta ante outliers y no linealidades.

También miramos la correlación absoluta con el target. Esto no decide por sí solo qué variables usar, pero ayuda a entender qué variables están más relacionadas con el evento de mora.

In [ ]:
feature_cols_for_corr = [c for c in train_fe.columns if c != TARGET]

corr_pearson = train_fe[[TARGET] + feature_cols_for_corr].corr(method="pearson")
corr_spearman = train_fe[[TARGET] + feature_cols_for_corr].corr(method="spearman")

target_corr = pd.DataFrame({
    "feature": feature_cols_for_corr,
    "pearson_with_target": corr_pearson.loc[feature_cols_for_corr, TARGET].values,
    "spearman_with_target": corr_spearman.loc[feature_cols_for_corr, TARGET].values,
})

target_corr["abs_pearson"] = target_corr["pearson_with_target"].abs()
target_corr["abs_spearman"] = target_corr["spearman_with_target"].abs()

target_corr.sort_values("abs_spearman", ascending=False).head(20)

In [ ]:
# Heatmap de correlación Spearman entre las principales variables por relación con el target.
top_corr_features = target_corr.sort_values("abs_spearman", ascending=False).head(15)["feature"].tolist()
plot_cols = [TARGET] + top_corr_features

plt.figure(figsize=(12, 10))
plt.imshow(train_fe[plot_cols].corr(method="spearman"), aspect="auto")
plt.colorbar(label="Correlación Spearman")
plt.xticks(range(len(plot_cols)), plot_cols, rotation=90)
plt.yticks(range(len(plot_cols)), plot_cols)
plt.title("Matriz de correlación Spearman - variables más relacionadas con el target")
plt.tight_layout()
plt.show()

In [ ]:
def high_correlation_pairs(corr_matrix: pd.DataFrame, threshold: float = 0.90) -> pd.DataFrame:
    """Devuelve pares de variables con correlación absoluta superior al umbral."""
    pairs = []
    cols = corr_matrix.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            value = corr_matrix.iloc[i, j]
            if abs(value) >= threshold:
                pairs.append({
                    "feature_1": cols[i],
                    "feature_2": cols[j],
                    "correlation": value,
                    "abs_correlation": abs(value),
                })
    return pd.DataFrame(pairs).sort_values("abs_correlation", ascending=False)

high_corr = high_correlation_pairs(train_fe[feature_cols_for_corr].corr(method="spearman"), threshold=0.90)
high_corr

## 11. Mutual Information

La correlación mide relaciones lineales o monótonas. `Mutual Information` puede detectar relaciones más generales entre una variable y el target. No presupone linealidad.

In [ ]:
# Mutual information requiere que no haya NaN. Ya imputamos previamente.
X_mi = train_fe[FINAL_FEATURES].astype(float)
y_mi = train_fe[TARGET].astype(int)

mi_values = mutual_info_classif(X_mi, y_mi, random_state=RANDOM_STATE)
mi_table = pd.DataFrame({"feature": FINAL_FEATURES, "mutual_information": mi_values})
mi_table = mi_table.sort_values("mutual_information", ascending=False)
mi_table.head(20)

In [ ]:
plt.figure(figsize=(10, 6))
top_mi = mi_table.head(20).iloc[::-1]
plt.barh(top_mi["feature"], top_mi["mutual_information"])
plt.title("Top 20 variables por Mutual Information con el target")
plt.xlabel("Mutual Information")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 12. PCA

PCA no se usará necesariamente en el modelo final, porque reduce interpretabilidad. Sin embargo, es útil para:

- detectar redundancia entre variables;
- visualizar si hay separación natural entre clases;
- entender cuánta varianza se concentra en pocos componentes.

Como PCA es sensible a escala, usamos `StandardScaler`.

In [ ]:
# Escalado de features para PCA.
pca_scaler = StandardScaler()
X_scaled = pca_scaler.fit_transform(train_fe[FINAL_FEATURES].astype(float))

pca = PCA(random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

explained = pd.DataFrame({
    "component": np.arange(1, len(pca.explained_variance_ratio_) + 1),
    "explained_variance_ratio": pca.explained_variance_ratio_,
})
explained["cumulative_explained_variance"] = explained["explained_variance_ratio"].cumsum()

explained.head(15)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(explained["component"], explained["cumulative_explained_variance"], marker="o")
plt.axhline(0.80, linestyle="--", linewidth=1)
plt.axhline(0.90, linestyle="--", linewidth=1)
plt.title("Varianza explicada acumulada por PCA")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza explicada acumulada")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Visualización 2D de los dos primeros componentes.
# Muestreamos para que el gráfico sea legible y rápido.
plot_sample = train_fe.sample(n=min(10000, len(train_fe)), random_state=RANDOM_STATE).index

plt.figure(figsize=(8, 6))
plt.scatter(
    X_pca[plot_sample, 0],
    X_pca[plot_sample, 1],
    c=train_fe.loc[plot_sample, TARGET],
    alpha=0.35,
    s=8,
)
plt.title("PCA: proyección en las dos primeras componentes")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="SeriousDlqin2yrs")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Loadings: peso de cada variable en cada componente.
loadings = pd.DataFrame(
    pca.components_.T,
    index=FINAL_FEATURES,
    columns=[f"PC{i}" for i in range(1, len(FINAL_FEATURES) + 1)]
)

# Variables con más peso absoluto en las primeras componentes.
for pc in ["PC1", "PC2", "PC3"]:
    print("\n", pc)
    display(loadings[pc].abs().sort_values(ascending=False).head(10).to_frame("abs_loading"))

## 13. Guardado de datasets y objetos

Guardamos:

- `preprocessed_train.csv`
- `preprocessed_prod.csv`
- lista de features finales
- imputador KNN
- límites de clipping
- resúmenes EDA

Los notebooks siguientes parten de estos ficheros para asegurar reproducibilidad.

In [ ]:
# Validaciones finales antes de guardar.
assert train_fe[TARGET].isna().sum() == 0, "El target de construcción no debe tener NaN."
assert prod_fe[TARGET].isna().sum() == len(prod_fe), "El target de producción debe permanecer vacío antes de predecir."
assert train_fe[FINAL_FEATURES].isna().sum().sum() == 0, "Quedan NaN en features de construcción."
assert prod_fe[FINAL_FEATURES].isna().sum().sum() == 0, "Quedan NaN en features de producción."

train_fe.to_csv(OUTPUT_DIR / "preprocessed_train.csv", index=False)
prod_fe.to_csv(OUTPUT_DIR / "preprocessed_prod.csv", index=False)

joblib.dump({
    "target": TARGET,
    "original_features": FEATURES,
    "final_features": FINAL_FEATURES,
    "missing_flag_cols": missing_flag_cols,
    "imputer": knn_imputer,
    "clipping_limits": clipping_limits,
    "pca_scaler": pca_scaler,
    "pca": pca,
}, OBJECTS_DIR / "preprocessing_objects.joblib")

mi_table.to_csv(OUTPUT_DIR / "mutual_information.csv", index=False)
target_corr.to_csv(OUTPUT_DIR / "target_correlations.csv", index=False)
high_corr.to_csv(OUTPUT_DIR / "high_correlations.csv", index=False)
explained.to_csv(OUTPUT_DIR / "pca_explained_variance.csv", index=False)

print("Ficheros guardados en:", OUTPUT_DIR.resolve())
print("Objetos guardados en:", OBJECTS_DIR.resolve())

## 14. Conclusiones del EDA y preprocesado

Puntos que deberías comentar en la memoria o presentación:

1. El problema está desbalanceado: hay muchos más clientes sin morosidad grave que con morosidad grave.
2. Las variables de retrasos pasados son candidatas muy fuertes para predecir el riesgo futuro.
3. Hay variables con colas extremadamente largas, especialmente ratios financieros. Por eso se ha aplicado clipping conservador.
4. Los valores perdidos se han tratado con KNN, pero conservando flags de ausencia para no perder información potencialmente relevante.
5. PCA se usa como análisis, no como transformación principal, porque el objetivo de la práctica incluye XAI y conviene conservar variables interpretables.
6. El dataset preprocesado queda listo para entrenar modelos de caja negra: red neuronal y aprendizaje por refuerzo contextual.